In [65]:
import cv2
import os
import numpy as np
from pathlib import Path
import glob
from ultralytics import YOLO

In [4]:
model = YOLO("yolov8x-seg.pt")

### All Images

In [61]:
test_images = []
test_labels = [] 
#read images from directory path 
for directory_path in glob.glob(r"C:\Users\Hp\Desktop\python\Computer Vision Projects\Final Project\pop cats\CNN code\PopCats Dataset\all images/*"):
    label = directory_path.split("\\")[-1]
    print("[STATUS] processed folder: {}".format(label))
    
    for img_path in glob.glob(os.path.join(directory_path, "*.jpg")):
       
        image = cv2.imread(img_path)
        # image = cv2.resize(image, fixed_size)

        test_images.append(image)
        test_labels.append(label)
        
print("[STATUS] completed Global Feature Extraction...")

[STATUS] processed folder: Abyssinian
[STATUS] processed folder: American Shorthair
[STATUS] processed folder: Bengal
[STATUS] processed folder: Birman
[STATUS] processed folder: Bombay
[STATUS] processed folder: Egyptian Mau
[STATUS] processed folder: Maine Coon
[STATUS] processed folder: Russian Blue
[STATUS] processed folder: Scottish Fold
[STATUS] processed folder: Sphynx
[STATUS] completed Global Feature Extraction...


In [62]:
print(len(test_images))

4999


In [63]:

def isolate_and_save_cats(images, labels, model, output_root_folder):

    # Loop through each image and its corresponding label
    for idx, (image, label) in enumerate(zip(images, labels)):
        
        # Predict using the YOLO model
        results = model.predict(image)

        # Create an output folder for the current label 
        output_label_folder = os.path.join(output_root_folder, label)
        os.makedirs(output_label_folder, exist_ok=True)

        # Iterate through the detection results 
        for r in results:
            # Iterate each object contour 
            for c in r:
                detected_label = c.names[c.boxes.cls.tolist().pop()]

                if detected_label.lower() == "cat":
                    b_mask = np.zeros(image.shape[:2], np.uint8)

                    # Create contour mask 
                    contour = c.masks.xy.pop().astype(np.int32).reshape(-1, 1, 2)
                    _ = cv2.drawContours(b_mask, [contour], -1, (255, 255, 255), cv2.FILLED)

                    # Isolate the cat with black background
                    mask3ch = cv2.cvtColor(b_mask, cv2.COLOR_GRAY2BGR)
                    isolated_cat = cv2.bitwise_and(mask3ch, image)

                    # Save the isolated cat image 
                    save_path = os.path.join(output_label_folder, f"{label}{idx}_isolated_cat.jpg")
                    cv2.imwrite(save_path, isolated_cat)

                    print(f"Isolated cat saved at: {save_path}")


In [64]:
output_root_folder = "isolated_cats_output"

isolate_and_save_cats(test_images, test_labels, model, output_root_folder)


0: 480x640 1 cat, 6926.9ms
Speed: 142.7ms preprocess, 6926.9ms inference, 53.4ms postprocess per image at shape (1, 3, 480, 640)
Isolated cat saved at: isolated_cats_output\Abyssinian\Abyssinian0_isolated_cat.jpg

0: 640x544 1 cat, 2716.3ms
Speed: 124.0ms preprocess, 2716.3ms inference, 7.0ms postprocess per image at shape (1, 3, 640, 544)
Isolated cat saved at: isolated_cats_output\Abyssinian\Abyssinian1_isolated_cat.jpg

0: 448x640 1 cat, 1 bed, 2147.1ms
Speed: 6.5ms preprocess, 2147.1ms inference, 9.2ms postprocess per image at shape (1, 3, 448, 640)
Isolated cat saved at: isolated_cats_output\Abyssinian\Abyssinian2_isolated_cat.jpg

0: 640x512 1 cat, 2 chairs, 2378.6ms
Speed: 3.0ms preprocess, 2378.6ms inference, 21.9ms postprocess per image at shape (1, 3, 640, 512)
Isolated cat saved at: isolated_cats_output\Abyssinian\Abyssinian3_isolated_cat.jpg

0: 640x480 1 cat, 2070.5ms
Speed: 71.3ms preprocess, 2070.5ms inference, 10.0ms postprocess per image at shape (1, 3, 640, 480)
Isol